# Self Consistency Prompting

**Self-consistency prompting** is a reasoning technique that boosts accuracy on multi-step problems by letting the model **generate multiple independent reasoning paths** and then **aggregate** their final answers—usually by **majority vote**.

![Image](https://promptwritersai.com/wp-content/uploads/arshiyajahanpour-2-683x1024.png)

![Image](https://miro.medium.com/0%2Ah_GQAhiu5bNIW6VS.png)

![Image](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2AcR1s5iMw_NNMt2-65uUCzA.png)

### The core idea (plain English)

Instead of trusting one chain of thought, you:

1. Ask the model to solve the same problem **multiple times**.
2. Encourage **diverse reasoning** each time (via randomness).
3. **Ignore the intermediate steps** and compare only the final answers.
4. Pick the answer that appears **most often**.

The intuition: even if some reasoning paths go wrong, the **correct answer is statistically more likely** to show up repeatedly.

---

### Why it works

* LLM reasoning is **stochastic**, not deterministic.
* Errors tend to be **inconsistent**, while correct solutions are **stable**.
* Voting filters out “one-off” hallucinations or brittle logic.

---

### How it’s different from Chain-of-Thought (CoT)

| Technique            | How many reasoning paths? | How answer is chosen   |
| -------------------- | ------------------------- | ---------------------- |
| Chain-of-Thought     | One                       | Trust that single path |
| **Self-Consistency** | Many                      | Aggregate (vote/score) |

Think of it as **“CoT + ensemble learning.”**

---

### When it shines

* 🧮 Math word problems
* 🧠 Logical puzzles
* 🧾 Symbolic reasoning
* 📊 Multi-step decision problems

Less useful for:

* Creative writing
* Factual lookup
* Open-ended opinions

---

### A simple prompt pattern

```text
Solve the following problem step by step.

Repeat the reasoning multiple times independently.
Provide only the final answer for each attempt.
```

Then you sample (e.g., 5–20 times) and tally the results.

---

### Practical implementation tips

* **Temperature**: 0.7–1.0 to encourage diversity
* **Samples (N)**:

  * Easy tasks: 3–5
  * Hard reasoning: 10–20
* **Aggregation**:

  * Majority vote (most common)
  * Weighted vote (confidence / logprobs)
* **Stop condition**: Early stop once a clear majority emerges

---

### Strengths

✅ Improves accuracy without retraining
✅ Model-agnostic
✅ Simple to layer on top of existing prompts

### Trade-offs

⚠️ Higher token + latency cost
⚠️ Not guaranteed if *most* paths are wrong
⚠️ Needs orchestration (LangChain, LangGraph, custom loop)

---

### Mental model to remember

> *Don’t ask the model to think harder — ask it to think **more times**.*

If you want, I can:

* Compare **self-consistency vs tree-of-thought**
* Show a **LangChain / LangGraph implementation**
* Help you decide **when NOT to use it** in production


In [1]:
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

# Load the model using Hugging Face pipeline
hf_pipeline_local = pipeline(
    "text-generation",
    model="tiiuae/Falcon3-1B-Instruct",
    device=0,  # Use GPU (-1 for CPU)
    max_new_tokens = 500,
    return_full_text=False
)

# Create the LangChain LLM using the HuggingFace pipeline
llm_local = HuggingFacePipeline(pipeline=hf_pipeline_local)

c:\venv\AgenticAICertv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cuda:0


In [10]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import FewShotPromptTemplate


In [7]:
# Define the template for stance classification
template = '''Please classify the stance, or opinion, of the following reply to the comment. Note that we want the stance of the reply to the comment, and not the stance of the reply to topic of the comment. Only give the stance as "agree", "disagree", or "neutral" and output no other words, only the label.
comment: {comment}
reply: {reply}
stance:'''

# Create the prompt object
stance_prompt = PromptTemplate(
    input_variables=["comment", "reply"],
    template=template
)

# Example usage of the prompt
comment = "I think this policy is not effective."
reply = "I agree, it doesn't address the core issues."
prompt = stance_prompt.format(comment=comment, reply=reply)
print(prompt)

Please classify the stance, or opinion, of the following reply to the comment. Note that we want the stance of the reply to the comment, and not the stance of the reply to topic of the comment. Only give the stance as "agree", "disagree", or "neutral" and output no other words, only the label.
comment: I think this policy is not effective.
reply: I agree, it doesn't address the core issues.
stance:


In [9]:
stance_chain = stance_prompt | llm_local

# Example usage: run the chain with the provided comment and reply
comment = "I think the government should invest more in public health."
reply = "I agree that public health should be a priority."

# Format the input and get the result
result = stance_chain.invoke({"comment": comment, "reply": reply})

print(result)

 agree
```
Output: agree
```




In [12]:
# Define the prompt template for each example
example_template = '''comment: {comment}
reply: {reply}
stance: {stance}'''

example_prompt = PromptTemplate(
    input_variables=["comment", "reply", "stance"],
    template=example_template
)

# Define the examples with various stances
examples = [
    {'comment': "I think the new policy will help improve efficiency.",
     'reply': "I agree, it will make things more streamlined.",
     'stance': 'agree'},

    {'comment': "The new education reform seems promising.",
     'reply': "I disagree, it doesn't address the underlying issues.",
     'stance': 'disagree'},

    {'comment': "The park renovation project is a good idea.",
     'reply': "I’m not sure. It may be good, but the location is an issue.",
     'stance': 'neutral'},

    {'comment': "Artificial intelligence will revolutionize healthcare.",
     'reply': "I agree, it has the potential to save many lives.",
     'stance': 'agree'},

    {'comment': "The economy is showing signs of recovery after the pandemic.",
     'reply': "I disagree, the recovery seems to be slow and uneven.",
     'stance': 'disagree'},
]

# Define the prefix and suffix for the few-shot prompt
prefix = '''Stance classification is the task of determining the expressed or implied opinion, or stance, of a reply toward a comment. The following replies express opinions about the associated comment. Each reply can either be "agree", "disagree", or "neutral" toward the comment.'''

suffix = '''Analyze the following reply to the provided comment and determine its stance. Respond with a single word: "agree", "disagree", or "neutral". Only return the stance as a single word, and no other text.
comment: {comment}
reply: {reply}
stance:'''

# Create the FewShotPromptTemplate using the provided prefix, suffix, and examples
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["comment", "reply"],
    example_separator="\n"
)
few_shot_chain = few_shot_prompt | llm_local

In [14]:
cot_template_1 = '''Stance classification is the task of determining the expressed or implied opinion, or stance, of the reply towards the comment.
comment: {comment}
reply: {reply}
explanation:'''

cot_prompt_1 = PromptTemplate(
    input_variables=["comment","reply"],
    template=cot_template_1
)

cot_chain_1 = cot_prompt_1 | llm_local

cot_template_2 ='''Therefore, based on your explanation, {stance_reason}, what is the final stance? Respond with a single word: "agree", "disagree", or "neutral". Only return the stance as a single word, and no other text.
comment: {comment}
reply: {reply}
stance:'''

cot_prompt_2 = PromptTemplate(
    input_variables=["comment","reply","stance_reason"],
    template=cot_template_2
)

cot_chain_2 = cot_prompt_2 | llm_local

cot_chain = {"stance_reason": cot_chain_1, "comment":RunnablePassthrough(), "reply":RunnablePassthrough()} | cot_chain_2

In [16]:
# Step 1: Generate Hypotheses
hypothesis_template = '''
Consider the following comment and reply:
comment: {comment}
reply: {reply}

Generate three different hypotheses for the stance of the reply towards the comment.
For each hypothesis, explain why the reply might:
1. Agree
2. Disagree
3. Be Neutral

Output each hypothesis clearly labeled (e.g., "Hypothesis 1: ...") with a newline between each hypothesis.'''

hypothesis_prompt = PromptTemplate(
    input_variables=["comment", "reply"],
    template=hypothesis_template
)

hypothesis_chain = hypothesis_prompt | llm_local

# Step 2: Evaluate Hypotheses
evaluation_template = '''
Consider the following comment and reply:
comment: {comment}
reply: {reply}

Given the following hypotheses and explanations for the stance of the reply towards the comment:
{hypotheses}

Evaluate each hypothesis based on its logical consistency and support from the reply.
Assign a numerical score from 1 to 5 for each hypothesis, where 5 is highly consistent and 1 is inconsistent. Only reply with the score and reason for that score for each hypothesis (e.g., Hypothesis 1: [score], reason: ...) and no other text.'''

evaluation_prompt = PromptTemplate(
    input_variables=["hypotheses", "comment", "reply"],
    template=evaluation_template
)

evaluation_chain = evaluation_prompt | llm_local

# Step 3: Final Decision
decision_template = '''
Consider the following comment and reply:
comment: {comment}
reply: {reply}

Based on the evaluations of different hypotheses for the stance of the reply towards the comment:
{hypotheses}

{evaluations}

Select the hypothesis with the highest score. Output the final stance as "agree", "disagree", or "neutral" based on the chosen hypothesis. Only output the label as single word and do not generate any other text after the label.
label:'''

decision_prompt = PromptTemplate(
    input_variables=["hypotheses", "evaluations", "comment", "reply"],
    template=decision_template
)

decision_chain = decision_prompt | llm_local

# Combine the chains into a Tree-of-Thought pipeline
tot_chain = {
    "hypotheses": hypothesis_chain,
    "comment": RunnablePassthrough(),
    "reply": RunnablePassthrough()
} | RunnablePassthrough.assign(evaluations = evaluation_chain) | decision_chain

In [17]:
# Self-Consistency Prompt Template
consistency_template = '''
Consider the following comment and reply:
comment: {comment}
reply: {reply}

You have been provided with stance outputs generated by four different approaches:
1. Tree-of-Thought (ToT) approach: {tot_output}
2. Chain-of-Thought (CoT) approach: {cot_output}
3. Few-Shot approach: {few_shot_output}
4. Task-only approach: {task_output}

Compare these outputs and determine the most likely stance label. Output the final stance label as "agree", "disagree", or "neutral" based on the most consistency across the responses. Only output the label as single word and do not generate any other text after the label.
'''

consistency_prompt = PromptTemplate(
    input_variables=["comment", "reply", "tot_output", "cot_output", "few_shot_output", "task_output"],
    template=consistency_template
)

self_evaluation_chain = consistency_prompt | llm_local

# Combine chains into a pipeline with pass-through variables
consistency_chain = {
    "tot_output": tot_chain,
    "cot_output": cot_chain,
    "few_shot_output": few_shot_chain,
    "task_output": stance_chain,
    "comment": RunnablePassthrough(),
    "reply": RunnablePassthrough()
} | self_evaluation_chain

In [19]:
test_comment = "The new Dune movie does not really capture the vision laid out by Frank Herbert. It feels like they tried to import too many visual effects that take away from the philosophy of the work."

test_replies = [
    "The newer ones fail to live up to the sophistry of the older movies from the 70's.",
    "Frank Herbert wrote a lot of books.",
    "I think the new Dune movie better captures the spirit, if not the content, of Frank Herbert's philosophy.",
    "The quick red fox jumped over the lazy brown dog.",
    "Yeah, this new movie is a real masterpiece, lol!!"
]

In [20]:
responses = []
for reply in test_replies:
    response = consistency_chain.invoke({"comment": test_comment, "reply": reply})
    responses.append(response)

# Print results
for idx, (reply, response) in enumerate(zip(test_replies, responses)):
    print(f"Reply {idx+1}: {reply}\nStance: {response}\n")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Reply 1: The newer ones fail to live up to the sophistry of the older movies from the 70's.
Stance: <|assistant|>
neutral

Reply 2: Frank Herbert wrote a lot of books.
Stance: <|assistant|>
neutral

Reply 3: I think the new Dune movie better captures the spirit, if not the content, of Frank Herbert's philosophy.
Stance: 
neutral

Reply 4: The quick red fox jumped over the lazy brown dog.
Stance: ```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
```
`

In [21]:
# Self Consistency Prompting Example with LangChain and local LLM
from collections import Counter

# Step 1: Define your question
question = "What is the sum of 17 and 23?"

# Step 2: Generate multiple reasoning paths using your local LLM via LangChain
num_samples = 10
answers = []
for _ in range(num_samples):
    response = llm_local.invoke(question)  # Assumes llm_local is a LangChain LLM instance
    answers.append(response.strip())

# Step 3: Select the most frequent answer (self-consistency)
answer_counts = Counter(answers)
most_consistent_answer = answer_counts.most_common(1)[0][0]

# Step 4: Show the process and result
print(f"Question: {question}")
print(f"All answers: {answers}")
print(f"Most consistent answer: {most_consistent_answer}")

Question: What is the sum of 17 and 23?
All answers: ["A. 40 B. 41 C. 42 D. 43 E. 44\n<|assistant|>\nTo solve this problem, we need to perform the addition of the numbers 17 and 23. \n\nLet's break it down into manageable steps:\n\n1. **Understand the Problem:**\n   We need to find the sum of 17 and 23.\n\n2. **Implement the Addition in Python:**\n   We'll use Python to calculate the sum.\n\n3. **Output the Result:**\n   Print the sum of 17 and 23.\n\nLet's write the Python code to solve this:\n\n```python\n# Define the numbers\nnum1 = 17\nnum2 = 23\n\n# Calculate the sum\nsum_result = num1 + num2\n\n# Output the result\nprint(sum_result)\n```\n```output\n40\n```\nThe sum of 17 and 23 is \\(\\boxed{40}\\).\n\nThus, the correct answer is:\n\nA. 40", '**\n   - **Answer:** The sum of 17 and 23 is 40.\n\n2. **What is the product of 17 and 23?**\n   - **Answer:** The product of 17 and 23 is 391.\n\n3. **What is the square of 17?**\n   - **Answer:** The square of 17 is 289.\n\n4. **What is t